# Writing a quantum program 3 — addressing bits

Swap two signed four-bit words by explicitly splitting them, swapping their corresponding MSB-first bits and joining them. Core owns the typed graph. Compiler owns construction and lowering. Simulation and Drawer consume that graph.

## Construct the program

Declare logical signed x and y words. Each split produces four independently owned bit bindings. Each bit swap consumes and returns the paired bindings. Join restores the original signed type. Bookkeeping contributes zero physical gates and zero workspace. The source code sets n=4 even though its comment says eight bits. This tutorial consistently uses four. The append adapter binds the stored typed graph to physical wires before composition.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from importlib import import_module

import numpy as np

import flagquantum as fq
from flagquantum.compiler.registers.component_builder import ComponentProgramBuilder
from flagquantum.compiler.registers.component_gate_function import bit_gate_function
from flagquantum.compiler.registers.component_graph import iter_component_calls
from flagquantum.compiler.registers.component_rebindings import (
    join_word_function,
    split_word_function,
)
from flagquantum.compiler.registers.primitives.swap import (
    source_swap_resource_formula,
    swap_bit_ir,
)
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.core.ir import CircuitIR
from flagquantum.core.ir.program import (
    FunctionRecord,
    ProgramRecord,
    verify_program_record,
)
from flagquantum.core.ir.program.register_bindings import RegisterBinding
from flagquantum.core.ir.program.register_types import (
    quantum_bit_type,
    signed_register_type,
)
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.simulation.classical_bit_binding import (
    join_classical_value,
    split_classical_value,
)
from flagquantum.simulation.classical_register_codec import (
    decode_register_bits,
    encode_register_bits,
)
from flagquantum.simulation.classical_swap import swap_classical_values
from flagquantum.simulation.component_classical_state import ComponentClassicalState


def make_swap_program(n: int = 4) -> ProgramRecord:
    signature = Signature(
        (Register("x", signed_register_type(n)), Register("y", signed_register_type(n)))
    )
    atom = bit_gate_function(
        "swap",
        (Register("x", quantum_bit_type()), Register("y", quantum_bit_type())),
        CircuitIR(2, swap_bit_ir(0, 1)),
    )
    builder = ComponentProgramBuilder(signature, (atom,))
    x, y = builder.inputs["x"], builder.inputs["y"]
    assert isinstance(x, RegisterBinding) and isinstance(y, RegisterBinding)
    xs, ys = builder.split(x), builder.split(y)
    for i in range(n):
        xs[i], ys[i] = builder.call_tuple("swap", {"x": xs[i], "y": ys[i]})
    return builder.finalize(
        {
            "x": builder.join(xs, signed_register_type(n)),
            "y": builder.join(ys, signed_register_type(n)),
        }
    )


@dataclass(frozen=True)
class WordSwap:
    """Append the physical circuit bound from the verified typed program."""

    n: int = 4

    def append_to(self, circuit: fq.Circuit) -> fq.Circuit:
        bound = bind_register_program(
            make_swap_program(self.n),
            (tuple(range(self.n)), tuple(range(self.n, 2 * self.n))),
        )
        return circuit.compose(bound.circuit)


n = 4
swap_routine = make_swap_program(n)
circuit = fq.Circuit(2 * n).append(WordSwap(n))


Verify the full module using Core verification.

In [ ]:
assert verify_program_record(swap_routine) == ()
circuit.to_ir().validate()


Draw the actual split, four swap and join calls from the stored graph. Every edge retains its typed producer and consumer.

In [ ]:
from flagquantum.drawer.call_graph import render_call_graph_svg
from flagquantum.drawer.component_graph import export_component_graph_dot

ipython_display = import_module("IPython.display")
graph_dot = export_component_graph_dot(swap_routine)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(graph_dot)))


The musical-score view keeps the four-bit boundaries, individual bit lanes and split/join connections. Drawer receives the same graph without executing its quantum operations.

In [ ]:
from examples.algorithms.classical_tensor_scores import symbols
from flagquantum.drawer.component_score import component_register_score
from flagquantum.drawer.register_score import RegisterStep
from flagquantum.drawer.register_score_data import (
    PortSide,
    SymbolClass,
    register_score_data,
)
from flagquantum.drawer.register_score_mpl import draw_register_score

base_symbols = symbols(swap_routine, {})


def swap_symbols(
    step: RegisterStep, register: str | None, side: PortSide | None
) -> tuple[SymbolClass, dict[str, str | int | bool]]:
    if step.name == "swap":
        if register is None:
            return "Text", {"text": "TwoBitSwap", "fontsize": 10}
        return "TextBox", {"text": "x" if register == step.inputs[0] else "y"}
    return base_symbols(step, register, side)


score = component_register_score(swap_routine)
score_data = register_score_data(score, wire_symbol=swap_symbols)
figure, axes = draw_register_score(score_data)
ipython_display.display(figure)
import_module("matplotlib.pyplot").close(figure)


## Classical simulation and resources

Evaluate each stored call using the shared split/join codecs and raw swap action. The example maps (5,6) to (6,5). The four actual SWAP primitives are reported independently of the model. The model has four Clifford gates, eight logical wires and zero T gates, measurements or work wires. No measurement branching occurs.

In [ ]:
def value_action(function: FunctionRecord, values: dict[str, object]) -> object:
    if function.name == "swap":
        x, y = swap_classical_values(values["x"], values["y"])
        return {"x": x, "y": y}
    signature = Signature(function.registers or ())
    left, right = tuple(signature.lefts()), tuple(signature.rights())
    if len(left) == len(right) == 1:
        if function == split_word_function(function.name, left[0].dtype):
            return {
                "reg": split_classical_value(
                    values["reg"],
                    lambda value: [
                        int(v)
                        for v in encode_register_bits(left[0], np.asarray(value)).flat
                    ],
                )
            }
        if function == join_word_function(function.name, right[0].dtype):
            return {
                "reg": join_classical_value(
                    np.asarray(values["reg"]),
                    lambda bits: decode_register_bits(right[0], np.asarray(bits)),
                )
            }
    raise ValueError(f"Unbound numerical swap recipe {function.name}")


def classical_swap(x: int, y: int) -> dict[str, object]:
    positions = tuple(i for i, _, _ in iter_component_calls(swap_routine))
    state = ComponentClassicalState(
        swap_routine, positions, {"x": x, "y": y}, value_action
    )
    return state.simulate()


result = classical_swap(5, 6)
print(result)
primitive_counts = Counter(g.name for g in circuit.to_ir().instructions)
model = source_swap_resource_formula(n)
print(dict(primitive_counts))
print(model)


## Provenance

Copyright 2024 Google LLC. SPDX-License-Identifier Apache-2.0. Adapted from Qualtran v0.7.0 `tutorials/quantum-programming/3-addressing-bits.ipynb`, revision `bcb17bb25360ed68ce3bc084645db3e929799389`, blob `a0f1acfbe5dd8c0bc05ced68b3255bcbcabad717`. All 11 original cells are mapped. Native builder and explicit append replace the source builder’s execution interface. The existing physical register binder executes the complete stored graph. Graphviz, IPython and Matplotlib are optional presentation dependencies.